In [ ]:
# Kiểm tra root_dir trên Kaggle
import os
print(os.listdir("/kaggle/input/datasets/nostagiguideus17"))


# Thiết lập để import source code
import sys
sys.path.append("/kaggle/input/datasets/nostagiguideus17/guru-legal-ai-retrieval")

# Cài đặt thêm lib cần thiết
!pip install -q bm25s
!pip install -q faiss-cpu
!pip install -q underthesea
!pip install -U bitsandbytes>=0.46.1

input_path = "/kaggle/input/datasets/nostagiguideus17/guru-legal-ai-retrieval/data"
output_path = "/kaggle/working"


In [ ]:
from kaggle_secrets import UserSecretsClient
HF_Token = UserSecretsClient().get_secret("HF_TOKEN")

import os
os.environ["HF_TOKEN"] = HF_Token

# 1. Data Cleaning & Format

In [ ]:
from src.preprocess_parquet import load

df = load(input_path + '/processed/combine_prunned.parquet')
# df = df.drop(columns=['article_title', 'source_note_text', 'source_links', 'topic_title'])
df

In [ ]:
df = df[df['article_index'].str.contains("Điều")]
df

# 2. Retrieving

In [ ]:
!pip install llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 -q



!mkdir -p /kaggle/working/models

!wget -q --show-progress \
    "https://huggingface.co/bartowski/Qwen2.5.1-Coder-7B-Instruct-GGUF/resolve/main/Qwen2.5.1-Coder-7B-Instruct-Q4_K_M.gguf" \
    -O /kaggle/working/models/Qwen2.5.1-Coder-7B-Instruct-Q4_K_M.gguf

MODEL_PATH = "/kaggle/working/models/Qwen2.5.1-Coder-7B-Instruct-Q4_K_M.gguf"

In [ ]:
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from src.precision_retriever import HyDEProcessor

# llm_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
# hf_token = os.environ.get("HF_TOKEN") # Đảm bảo bạn đã export token này

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16
# )

# # Thêm tham số token=hf_token
# llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name, token=hf_token)
# llm_model = AutoModelForCausalLM.from_pretrained(
#     llm_model_name,
#     quantization_config=bnb_config,
#     device_map={"": 0},
#     token=hf_token
# )

In [ ]:
from src.precision_retriever import LlamaHyDEProcessor

print("[INFO] Đang nạp Local LLM (4-bit) cho HyDE...")


# Khởi tạo HyDE
hyde_processor = LlamaHyDEProcessor(
    model_path = MODEL_PATH,
    main_gpu=0,
    n_ctx = 4096,
    max_new_tokens = 256,
    system_prompt = """
    Bạn là một chuyên gia phân tích pháp lý, chịu trách nhiệm Phân rã truy vấn (Query Decomposition) và Xây dựng văn bản quy phạm giả định (Hypothetical Document) nhằm tối ưu hóa việc truy hồi tài liệu pháp luật Việt Nam. 

    Nhiệm vụ của bạn là chuyển đổi câu hỏi mang tính chất thắc mắc của người dùng thành một đoạn văn bản mô tả giả định có văn phong, cấu trúc tương đương với các nghị định, thông tư hoặc điều luật.
    
    
    Đoạn văn bản giả định bắt buộc phải trích xuất và thể hiện được các yếu tố sau:
    1. Đối tượng (Chủ thể): Xác định rõ các cá nhân, cơ quan, tổ chức hoặc thực thể có liên quan trực tiếp đến tình huống.
    2. Ý định: Thể hiện rõ mục đích, nguyện vọng, hoặc nhu cầu cốt lõi của đối tượng.
    3. Hành vi/Nghĩa vụ: Mô tả cụ thể các hành động, trách nhiệm, quy trình hoặc quyền lợi mà đối tượng phải thực hiện, được phép thực hiện, hoặc bị nghiêm cấm.
    
    Yêu cầu nghiêm ngặt về mặt thể hiện:
    - Văn phong pháp lý: Sử dụng hoàn toàn câu tường thuật, câu khẳng định hoặc câu điều kiện hành chính (Ví dụ sử dụng các cấu trúc: "Trường hợp cá nhân có nguyện vọng...", "Chủ thể có nghĩa vụ...", "Cơ quan có thẩm quyền thực hiện..."). 
    - Không đặt câu hỏi: Tuyệt đối không dùng câu nghi vấn, không đặt câu hỏi ngược lại, không có dấu chấm hỏi.
    - Kiểm soát thông tin (Chống nhiễu truy hồi): Chỉ sử dụng các thuật ngữ có sẵn trong câu hỏi gốc. Tuyệt đối KHÔNG tự ý đưa vào tên các bộ luật cụ thể (như Luật Dân sự, Luật Đất đai...), không tự bịa số hiệu điều/khoản, và không tự thêm các số liệu, thời gian, thời hạn cụ thể nằm ngoài câu hỏi. Thay vào đó, hãy dùng từ ngữ tổng quát như "theo quy định của pháp luật", "trong thời hạn quy định", "tại cơ quan có thẩm quyền".
    - Trình bày mỗi nhóm quan hệ Chủ thể - Ý định - Hành vi thành một câu đơn độc lập.
    - Ngôn ngữ: Chỉ sử dụng Tiếng Việt.
    - Định dạng đầu ra: Chỉ trả về duy nhất một đoạn văn bản giả định (khoảng 50 - 100 từ). Tuyệt đối không chào hỏi, không giải thích lý do, không thêm nhận xét hay bất kỳ ký tự dẫn chuyện nào khác.    
    """
)

print("[SUCCESS] Local HyDE đã sẵn sàng.")

In [ ]:
print(hyde_processor.enhance(
    "Các cơ sở ươm tạo và khu làm việc chung được hưởng những chính sách hỗ trợ nào về thuế và đất đai"
))

In [ ]:
from src.recall_retriever import BM25
from src.data_presentation import Corpus, HierarchicalCorpus

corpus_simple = Corpus()
corpus_hier = HierarchicalCorpus(
    alpha = 0.5,
    title_blending = True
)

bm25_raw = BM25(
    name="BM25_ra",
    corpus= corpus_simple,
    top_k=500,
    norm=0.75,
    use_segmentation=True
)
bm25_hier = BM25(
    name="BM25_hi",
    corpus= corpus_hier,
    top_k=4000,     # vì top-K này dựa trên bản Articles bị chunk (mảnh) rồi. Nên để top_k lớn hơn ít nhất 6-10 lần số lượng cần lấy, để đảm bảo đúng ranking.
    norm=1.0,
    use_segmentation=True
)


In [ ]:
from sentence_transformers import SentenceTransformer
from src.recall_retriever import Dense
import torch

dense_model = SentenceTransformer(
    "keepitreal/vietnamese-sbert",
    token=os.environ.get("HF_TOKEN"),
    model_kwargs={"torch_dtype": torch.float16},
    device="cuda:1"
)
print(f"[SUCCESS] Dense Model sẵn sàng trên: {dense_model.device}")


dense_raw = Dense(
    corpus_simple, dense_model,
    batch_size=8,
    top_k = 200, 
    M=32, ef_construction=512, ef_search=256,
    index_type='Flat', name='DENSE_ra' 
)


dense_hier = Dense(
    corpus_hier, dense_model,
    batch_size=2,
    top_k = 2048, 
    M=256, ef_construction=4096, ef_search=2048,
    index_type='HNSW', name='DENSE_hi' 
)

In [ ]:
from sentence_transformers import CrossEncoder
from src.precision_retriever import CrossEncoderReranker



reranker_model = CrossEncoder(
    "BAAI/bge-reranker-v2-m3", 
    activation_fn=torch.nn.Identity(),
    model_kwargs={"torch_dtype": torch.float16},
    device="cuda:1"
)
print(f"[SUCCESS] Cross-Encoder sẵn sàng trên: {reranker_model.model.device}")

# Khởi tạo class
crossEncoder = CrossEncoderReranker(
    model=reranker_model,
    name="CE",
    batch_size=4
)

In [ ]:
from src.retrieval_pipeline import RetrievalPipeline


retriever = RetrievalPipeline(
    data = df['content_text'],
    HyDE = hyde_processor,
    recallLayers=[dense_hier, bm25_hier, bm25_raw],
    precisionLayers=[crossEncoder],
    top_re_rank=128,
    fusion_target = {
        'DENSE_hi_score': 4.5,
        'BM25_hi_score': 3.0,
        'BM25_ra_score': 2.5
    },
    save_folder="/kaggle/input/datasets/nostagiguideus17/guru-legal-ai-retrieval/saved_model/vbpl"
)

Bao giờ chạy xong thì save lại model đã fit với KB. Lần sau có thể down lại sử dụng (không phải chạy lại Cell trên nữa).
Muốn thế thì thay tên `save_folder` vào nha.

In [ ]:
# retriever.save("/kaggle/working/saved_model")

# 3. Manual Evaluation

In [ ]:
import pandas as pd

score = retriever.retrieve("Đối tác chậm giao hàng khiến công ty không đạt được mục đích kinh doanh, công ty muốn chấm dứt hợp đồng thì nên chọn phương án hủy bỏ hay đơn phương chấm dứt để xử lý việc hoàn trả tài sản và bồi thường thiệt hại như thế nào?")
  
score_board = pd.concat([df[['docs_title', 'article_index']], score], axis=1)

score_board.sort_values(by="CE_Robust_score", ascending=False).head(20).tail(20)

# 3. Run test

In [ ]:
unused_mask = pd.Series(True, index=df.index)

In [ ]:
import json
import copy
from contextlib import ExitStack
import time
import os  # Đảm bảo đã import os

data = input_path + '/qa/R2AIStage1DATA.json'
result_path = output_path + '/results'
qa_range = range(667,1333)

# Đánh dấu thời gian bắt đầu
start_time = time.perf_counter()

# ==================== CẤU HÌNH TẠI ĐÂY ====================
TOP_K_LIST = [3, 4, 5, 10, 20, 50, 100]
SCORE_COLUMNS = {
    "bm25_raw": "BM25_ra_score",
    "bm25_hier": "BM25_hi_score",
    "rrf": "RRF_score",
    # "dense_raw": "DENSE_ra_score",
    "dense_hier": "DENSE_hi_score",
    "CE_Sigmoid": "CE_Sigmoid_score",
    "CE_Robust": "CE_Robust_score",
    "CE_Gap": "CE_Gap_score",
    "CE_Z_score": "CE_Z_score",
    "CE_MinMax_score": "CE_MinMax_score",
}
# Cấu hình danh sách các ngưỡng Threshold cho Cross Encoder
CE_THRESHOLDS = [0.8, 0.85, 0.87, 0.9, 0.92, 0.95, 0.975, 1.0, 1.05, 1.1, 1.15]
# Lọc nhanh các key bắt đầu bằng CE phục vụ cho việc tạo file và vòng lặp
CE_METRICS = [key for key in SCORE_COLUMNS.keys() if key.startswith("CE")]
# ==========================================================

# 1. Đọc dữ liệu một lần duy nhất (Load vào bộ nhớ)
try:
    with open(data, 'r', encoding='utf-8') as f_in:
        data = json.load(f_in)
        
        if isinstance(data, dict):
            data = [data]
except FileNotFoundError:
    print(f"Không tìm thấy file")
    data = []

# 2. TẠO THƯ MỤC TRƯỚC KHI LƯU FILE
os.makedirs(f"{result_path}/submit", exist_ok=True)
os.makedirs(f"{result_path}/index", exist_ok=True)

with ExitStack() as stack:

    # File handlers cho file chính (chứa relevant_docs, relevant_articles)
    file_handlers_main = {
        metric: {
            k: stack.enter_context(
                open(
                    f"{result_path}/submit/{metric}_top{k}.jsonl",
                    "a",
                    encoding="utf-8"
                )
            )
            for k in TOP_K_LIST
        }
        for metric in SCORE_COLUMNS
    }

    # File handlers cho file index bổ sung (chứa articles_loc)
    file_handlers_index = {
        metric: {
            k: stack.enter_context(
                open(
                    f"{result_path}/index/{metric}_top{k}_index.jsonl",
                    "a",
                    encoding="utf-8"
                )
            )
            for k in TOP_K_LIST
        }
        for metric in SCORE_COLUMNS
    }

    # ================= ĐÃ SỬA: KHỞI TẠO FILE HANDLERS CHO TỪNG LOẠI CE + THRESHOLD =================
    file_handlers_ce_submit = {
        metric: {
            thresh: stack.enter_context(
                open(f"{result_path}/submit/{metric}_threshold{thresh}.jsonl", "a", encoding="utf-8")
            )
            for thresh in CE_THRESHOLDS
        }
        for metric in CE_METRICS
    }
    
    file_handlers_ce_index = {
        metric: {
            thresh: stack.enter_context(
                open(f"{result_path}/index/{metric}_threshold{thresh}_index.jsonl", "a", encoding="utf-8")
            )
            for thresh in CE_THRESHOLDS
        }
        for metric in CE_METRICS
    }
    # ==============================================================================================

    for idx, item in enumerate(data):
        if idx not in qa_range: continue

        query = item["question"]
        print(f"Start [{idx}/{len(data)}]: {query}")

        score = retriever.retrieve(query)

        # 1. Xử lý các file Top K thông thường (giữ nguyên)
        for metric, column in SCORE_COLUMNS.items():
            sorted_score = score.sort_values(
                column,
                ascending=False
            )

            unused_mask[sorted_score.head(2000).index] = False

            for k in TOP_K_LIST:
                top_k_idx = sorted_score.head(k).index
                articles_df = df.loc[top_k_idx]

                # --- XỬ LÝ CHO FILE CHÍNH ---
                item_main = copy.deepcopy(item)
                item_main["answer"] = ""

                item_main["relevant_docs"] = (
                    (articles_df["docs_code"] + "|" + articles_df["docs_title"])
                    .unique()
                    .tolist()
                )

                item_main["relevant_articles"] = (
                    (articles_df["docs_code"] + "|" + articles_df["docs_title"] + "|" + articles_df["article_index"])
                    .tolist()
                )

                # --- GHI VÀO FILE CHÍNH ---
                f_out_main = file_handlers_main[metric][k]
                json_str_main = json.dumps(item_main, ensure_ascii=False)
                f_out_main.write(json_str_main + "\n")
                f_out_main.flush()
                
                # --- XỬ LÝ CHO FILE INDEX ---
                item_index = copy.deepcopy(item)
                item_index['articles_loc'] = top_k_idx.tolist()

                # --- GHI VÀO FILE INDEX ---
                f_out_index = file_handlers_index[metric][k]
                json_str_index = json.dumps(item_index, ensure_ascii=False)
                f_out_index.write(json_str_index + "\n")
                f_out_index.flush()

        # ================= ĐÃ SỬA: XỬ LÝ LƯU THEO LIST NGƯỠNG CHO TẤT CẢ KEY CE =================
        if retriever.precisionLayers is not None:
            for ce_metric in CE_METRICS:
                column = SCORE_COLUMNS[ce_metric]
                
                # Sắp xếp theo score của key CE hiện tại
                ce_sorted = score.sort_values(column, ascending=False)
                
                # Lặp qua từng ngưỡng trong list
                for thresh in CE_THRESHOLDS:
                    # Lọc lấy những index có điểm CE >= threshold hiện tại
                    threshold_mask = ce_sorted[column] >= thresh
                    ce_threshold_idx = ce_sorted[threshold_mask].index
    
                    articles_ce_df = df.loc[ce_threshold_idx]
    
                    # --- Ghi vào file Submit Threshold ---
                    item_ce_submit = copy.deepcopy(item)
                    item_ce_submit["answer"] = ""
                    item_ce_submit["relevant_docs"] = (articles_ce_df["docs_code"] + "|" + articles_ce_df["docs_title"]).unique().tolist()
                    item_ce_submit["relevant_articles"] = (articles_ce_df["docs_code"] + "|" + articles_ce_df["docs_title"] + "|" + articles_ce_df["article_index"]).tolist()
                    
                    f_out_ce_submit = file_handlers_ce_submit[ce_metric][thresh]
                    f_out_ce_submit.write(json.dumps(item_ce_submit, ensure_ascii=False) + "\n")
                    f_out_ce_submit.flush()
    
                    # --- Ghi vào file Index Threshold ---
                    item_ce_index = copy.deepcopy(item)
                    item_ce_index['articles_loc'] = ce_threshold_idx.tolist()
                    
                    f_out_ce_index = file_handlers_ce_index[ce_metric][thresh]
                    f_out_ce_index.write(json.dumps(item_ce_index, ensure_ascii=False) + "\n")
                    f_out_ce_index.flush()
        # =========================================================================================

    print("Đã hoàn tất quá trình cập nhật và lưu file!")

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Thời gian chạy: {execution_time:.6f} giây")

In [ ]:
black_list = df[unused_mask][['docs_code', 'docs_title', 'article_index', 'subject_title']]
black_list.to_csv(output_path + "/blacklist_articles.csv")

black_list